# Train Camo/Gun Detector on Google Colab

Free T4 GPU, no local setup. Steps:

1. **Runtime → Change runtime type → GPU (T4)**
2. Run the cells below in order.
3. Upload your dataset zip when prompted (or mount Google Drive).

Expects a zip with this structure:
```
dataset.zip
├── images/{train,val}/*.jpg
├── labels/{train,val}/*.txt   # YOLO format
└── data.yaml                   # optional; we'll write one if missing
```

In [ ]:
# 1. Verify GPU
!nvidia-smi

In [ ]:
# 2. Install Ultralytics (ships YOLO26 + YOLO11)
%pip install -q --upgrade ultralytics opencv-python-headless
import ultralytics; ultralytics.checks()

In [ ]:
# 3. Clone the repo
!git clone https://github.com/ChrisPuzzo/YOLOv7-Camo-Detection.git
%cd YOLOv7-Camo-Detection
!git checkout feat/yolo26-rebuild

In [ ]:
# 4. Pull pre-labeled camo datasets from Roboflow Universe.
# Get a free API key at https://app.roboflow.com/settings/api
import os, getpass
os.environ['ROBOFLOW_API_KEY'] = getpass.getpass('Roboflow API key: ')
%pip install -q roboflow
!python scripts/fetch_datasets.py

# Alternative: upload your own dataset.zip from your computer
# from google.colab import files
# uploaded = files.upload()
# !unzip -q -o dataset.zip -d dataset/

!ls dataset/images/train/ 2>/dev/null | head
!ls dataset/labels/train/ 2>/dev/null | head

In [ ]:
# 5. Train. Adjust epochs / imgsz as needed.
!python scripts/train.py \
    --model yolo26n.pt \
    --epochs 100 \
    --imgsz 640 \
    --batch 16 \
    --name camo_beta_colab

In [ ]:
# 6. Evaluate
!python scripts/eval.py --weights models/best.pt

In [ ]:
# 7. Export for mobile (TFLite INT8 for Android, ONNX for cross-platform).
# CoreML can only be exported on macOS, so do that step locally on a Mac.
!python scripts/export_mobile.py --weights models/best.pt --formats tflite onnx

In [ ]:
# 8. Download the trained weights + mobile artifacts
from google.colab import files
import os
for f in ['models/best.pt', 'models/best_int8.tflite', 'models/best.onnx']:
    if os.path.exists(f):
        files.download(f)